# Install Dependecies

In [28]:
%%capture
%pip install numpy
%pip install pandas
%pip install matplotlib.pyplot
%pip install python-terrier

In [29]:
%%capture
!curl -s "https://get.sdkman.io" | bash && source "$HOME/.sdkman/bin/sdkman-init.sh" && sdk install java 11.0.22-amzn < /dev/null

# check java and version
!which java
!java -version
!readlink -f $(which java)
!ls -la /usr/lib/jvm
!java --version
!javac --version

# Imports

In [30]:
import itertools
import json
import os
import re
import time
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pyterrier as pt
from pathlib import Path

# PyTerrier - Local

In [31]:
# os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
# os.environ["JVM_PATH"] = "/usr/lib/jvm/java-11-openjdk-amd64/lib/server/libjvm.so"
# os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

# import pyterrier as pt

# if not pt.java.started():
#     pt.java.init()

# print("JAVA_HOME:", os.environ["JAVA_HOME"])
# print("JVM_PATH:", os.environ["JVM_PATH"])
# print("Java started:", pt.java.started())

# PyTerrier - Colab

In [32]:
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

if not pt.started():
    pt.init()

print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("Java started:", pt.java.started())

JAVA_HOME: /usr/lib/jvm/java-17-openjdk-amd64
Java started: True


/tmp/ipykernel_17549/631900236.py:4: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():


# Load Dataset

In [33]:
base = 'https://raw.githubusercontent.com/Legenden84/search-engines/master/project_handout'

docs    = pd.read_json(f'{base}/docs2.jsonl', lines=True, dtype={'docno': str})
train_queries = pd.read_csv(f'{base}/train_queries.csv')
train_qrels   = pd.read_csv(f'{base}/train_qrels.csv')

# Load Indexes

In [40]:
index_path_none = "./indexes/full"
index_path_stop = "./indexes/stopwords"
index_path_stem = "./indexes/stemming"
index_path_stop_stem = "./indexes/stop-stem"

In [44]:
index_none = pt.IndexFactory.of(index_path_none)
index_stop = pt.IndexFactory.of(index_path_stop)
index_stem = pt.IndexFactory.of(index_path_stem)
index_stop_stem = pt.IndexFactory.of(index_path_stop_stem)

indices = {
    "stopwords": index_stop,
    "stop_stem": index_stop_stem,

    # Optional
    "none": index_none,
    "stem": index_stem
}

# Preprocessing

In [45]:
if "text" in train_queries.columns and "query" not in train_queries.columns:
    train_queries = train_queries.rename(columns={"text": "query"})

train_queries["qid"] = train_queries["qid"].astype(str)
train_queries["query"] = train_queries["query"].astype(str)

train_qrels["qid"] = train_qrels["qid"].astype(str)
train_qrels["docno"] = train_qrels["docno"].astype(str)

if "label" not in train_qrels.columns:
    if "relevance" in train_qrels.columns:
        train_qrels = train_qrels.rename(columns={"relevance": "label"})
    elif "rel" in train_qrels.columns:
        train_qrels = train_qrels.rename(columns={"rel": "label"})

train_qrels["label"] = train_qrels["label"].astype(int)

print("Training queries:")
display(train_queries.head())

print("Training qrels:")
display(train_qrels.head())

Training queries:


,qid,query
0,0,when did richmond last play in a preliminary f...
1,1,who sang what in the world's come over you
2,2,who produces the most wool in the world
3,3,where does alaska the last frontier take place
4,4,a day to remember all i want cameos


Training qrels:


,qid,docno,label,iteration
0,5743,D2,2,0
1,6266,D13,2,0
2,7918,D17,2,0
3,5215,D25,1,0
4,1173,D37,1,0


# Helpers

In [46]:
def evaluate_model_for_tuning(model, queries_df, qrels_df, measure="ndcg_cut_10"):
    result = pt.Experiment(
        retr_systems=[model],
        topics=queries_df,
        qrels=qrels_df,
        eval_metrics=[measure],
        names=["model"],
        filter_by_qrels=True,
    )
    return result.iloc[0][measure]

# Tuning Constants

In [47]:
EVAL_MEASURE = "ndcg_cut_10"

bm25_grid = {
    "k1": [0.6, 0.9, 1.2, 1.5, 2.0],
    "b":  [0.2, 0.4, 0.6, 0.75, 0.9]
}

lm_grid = {
    "c": [0.05, 0.10, 0.15, 0.20, 0.30, 0.50]
}

cache_dir = Path("./results")
cache_dir.mkdir(parents=True, exist_ok=True)

bm25_cache_path = cache_dir / "bm25_tuning_results.json"
lm_cache_path   = cache_dir / "lm_tuning_results.json"

tuning_results = []
best_models = {
    index_name: {}
    for index_name in indices.keys()
}

# 3 Ranking Models and Evaluation

Once you have the index, you can start implementing your ranking models. You should do the following: Tune and run BM25 and a Language Model (LM). Given the time requirements, we require that you do this only for the two indices with the stopwords removed16, separately. You are allowed to use existing search libraries. The same library that you have used to build your index will likely have a version of BM25 and LM. To tune your ranking models, you should use all your training queries. After finding the best configuration for your model, you should use that model (tuned model) on the unseen queries that we will release on week 21. You should tune with respect to one evaluation measure only (either MRR or NDCG@10). We require that you use trec-eval17 for evaluation.

# 3.1 Ranking Models

Present in a table, for each ranking model, and for each version of the index with the stopwords removed, NDCG, MRR, Precision and Recall at 5, 10 and 20 cutoffs, and mean response time (the time of getting the evaluation score of a query). If you split the training queries (for example into train and validation sets), present the results of the above metrics on each of the sets.

### 3.1.1 BM25 Tuning

In [48]:
def load_bm25_cache():
    """Returns (tuning_results, best_configs) from the cache file.
    ([], {}) if the cache does not exist yet."""
    if not bm25_cache_path.exists():
        return [], {}
    with open(bm25_cache_path, "r") as f:
        cache = json.load(f)
    return cache["tuning_results"], cache["best_configs"]

In [49]:
def sync_bm25_state_from_cache():
    """Refresh shared state (`tuning_results`, `best_models`) from the cache file.
    Idempotent."""
    cached_rows, cached_best = load_bm25_cache()

    # Replace BM25 rows in the shared tuning_results list.
    tuning_results[:] = [r for r in tuning_results if r.get("model") != "BM25"]
    tuning_results.extend(cached_rows)

    # Rebuild best_models[<index>]["BM25"] from the cache.
    for index_name, best_info in cached_best.items():
        if index_name not in indices:
            continue
        index  = indices[index_name]
        config = best_info["config"]
        best_models[index_name]["BM25"] = {
            "model": pt.terrier.Retriever(
                index,
                wmodel="BM25",
                controls={
                    "bm25.k_1": config["k1"],
                    "bm25.b":   config["b"]
                }
            ),
            "config": config,
            "score":  best_info["score"]
        }

In [50]:
def tune_bm25_for_index(index_name, force=False):
    """Tune BM25 (k1, b) for a single index. Writes results to the cache file
    and syncs shared state. If the index is already in the cache, this is a
    no-op unless force=True. Returns nothing — use load_bm25_cache() to inspect."""
    if index_name not in indices:
        raise ValueError(
            f"Unknown index name: {index_name!r}. "
            f"Available: {list(indices.keys())}"
        )

    index   = indices[index_name]
    configs = list(itertools.product(bm25_grid["k1"], bm25_grid["b"]))
    total   = len(configs)

    if bm25_cache_path.exists():
        with open(bm25_cache_path, "r") as f:
            cache = json.load(f)
    else:
        cache = {
            "eval_measure":   EVAL_MEASURE,
            "grid":           bm25_grid,
            "tuning_results": [],
            "best_configs":   {}
        }

    # Skip if this index is already tuned (unless force=True).
    if index_name in cache["best_configs"] and not force:
        print(
            f"BM25 for {index_name!r} already in cache "
            f"({cache['best_configs'][index_name]['config']}, "
            f"{EVAL_MEASURE}={cache['best_configs'][index_name]['score']:.4f}). "
            f"Skipping. Pass force=True to retune."
        )
        sync_bm25_state_from_cache()
        return

    # Drop any prior rows for this index (re-tuning with force=True).
    cache["tuning_results"] = [
        r for r in cache["tuning_results"] if r.get("index") != index_name
    ]

    best_score  = -1
    best_config = None

    for step, (k1, b) in enumerate(configs, start=1):
        print(
            f"[{step:>2}/{total}] index={index_name} k1={k1}, b={b}",
            flush=True
        )

        model = pt.terrier.Retriever(
            index,
            wmodel="BM25",
            controls={
                "bm25.k_1": k1,
                "bm25.b":   b
            }
        )
        score = evaluate_model_for_tuning(
            model, train_queries, train_qrels, measure=EVAL_MEASURE
        )

        cache["tuning_results"].append({
            "index": index_name,
            "model": "BM25",
            "k1":    k1,
            "b":     b,
            "c":     None,
            EVAL_MEASURE: score
        })

        if score > best_score:
            best_score  = score
            best_config = {"k1": k1, "b": b}

    cache["best_configs"][index_name] = {
        "config": best_config,
        "score":  best_score
    }

    with open(bm25_cache_path, "w") as f:
        json.dump(cache, f, indent=4)

    print(f"Saved BM25 tuning for {index_name} to {bm25_cache_path}")
    sync_bm25_state_from_cache()


if bm25_cache_path.exists():
    sync_bm25_state_from_cache()
    print(f"Loaded existing BM25 cache from {bm25_cache_path}")
else:
    print(
        f"No BM25 cache yet at {bm25_cache_path}.\n"
        f"Available indexes to tune: {list(indices.keys())}\n"
        f"Example:  tune_bm25_for_index('stopwords')"
    )

Loaded existing BM25 cache from results/bm25_tuning_results.json


In [54]:
tune_bm25_for_index("stopwords")

BM25 for 'stopwords' already in cache ({'k1': 0.9, 'b': 0.6}, ndcg_cut_10=0.4434). Skipping. Pass force=True to retune.


In [56]:
tune_bm25_for_index("stop_stem")

[ 1/25] index=stop_stem k1=0.6, b=0.2
[ 2/25] index=stop_stem k1=0.6, b=0.4
[ 3/25] index=stop_stem k1=0.6, b=0.6
[ 4/25] index=stop_stem k1=0.6, b=0.75
[ 5/25] index=stop_stem k1=0.6, b=0.9
[ 6/25] index=stop_stem k1=0.9, b=0.2
[ 7/25] index=stop_stem k1=0.9, b=0.4
[ 8/25] index=stop_stem k1=0.9, b=0.6
[ 9/25] index=stop_stem k1=0.9, b=0.75
[10/25] index=stop_stem k1=0.9, b=0.9
[11/25] index=stop_stem k1=1.2, b=0.2
[12/25] index=stop_stem k1=1.2, b=0.4
[13/25] index=stop_stem k1=1.2, b=0.6
[14/25] index=stop_stem k1=1.2, b=0.75
[15/25] index=stop_stem k1=1.2, b=0.9
[16/25] index=stop_stem k1=1.5, b=0.2
[17/25] index=stop_stem k1=1.5, b=0.4
[18/25] index=stop_stem k1=1.5, b=0.6
[19/25] index=stop_stem k1=1.5, b=0.75
[20/25] index=stop_stem k1=1.5, b=0.9
[21/25] index=stop_stem k1=2.0, b=0.2
[22/25] index=stop_stem k1=2.0, b=0.4
[23/25] index=stop_stem k1=2.0, b=0.6
[24/25] index=stop_stem k1=2.0, b=0.75
[25/25] index=stop_stem k1=2.0, b=0.9
Saved BM25 tuning for stop_stem to results/bm

### 3.1.2 LM Tuning

In [57]:
def load_lm_cache():
    """Returns (tuning_results, best_configs) from the LM cache file.
    ([], {}) if the cache does not exist yet."""
    if not lm_cache_path.exists():
        return [], {}
    with open(lm_cache_path, "r") as f:
        cache = json.load(f)
    return cache["tuning_results"], cache["best_configs"]

In [58]:
def sync_lm_state_from_cache():
    """Refresh shared state (`tuning_results`, `best_models`) from the LM cache file.
    Idempotent."""
    cached_rows, cached_best = load_lm_cache()

    # Replace Hiemstra_LM rows in the shared tuning_results list.
    tuning_results[:] = [r for r in tuning_results if r.get("model") != "Hiemstra_LM"]
    tuning_results.extend(cached_rows)

    # Rebuild best_models[<index>]["Hiemstra_LM"] from the cache.
    for index_name, best_info in cached_best.items():
        if index_name not in indices:
            continue
        index  = indices[index_name]
        config = best_info["config"]
        best_models[index_name]["Hiemstra_LM"] = {
            "model": pt.terrier.Retriever(
                index,
                wmodel="Hiemstra_LM",
                controls={
                    "c": config["c"]
                }
            ),
            "config": config,
            "score":  best_info["score"]
        }

In [59]:
def tune_lm_for_index(index_name, force=False):
    """Tune Hiemstra_LM (c) for a single index. Writes results to the cache
    file and syncs shared state. If the index is already in the cache, this
    is a no-op unless force=True. Returns nothing — use load_lm_cache() to
    inspect."""
    if index_name not in indices:
        raise ValueError(
            f"Unknown index name: {index_name!r}. "
            f"Available: {list(indices.keys())}"
        )

    index   = indices[index_name]
    configs = list(lm_grid["c"])
    total   = len(configs)

    if lm_cache_path.exists():
        with open(lm_cache_path, "r") as f:
            cache = json.load(f)
    else:
        cache = {
            "eval_measure":   EVAL_MEASURE,
            "grid":           lm_grid,
            "tuning_results": [],
            "best_configs":   {}
        }

    # Skip if this index is already tuned (unless force=True).
    if index_name in cache["best_configs"] and not force:
        print(
            f"Hiemstra_LM for {index_name!r} already in cache "
            f"({cache['best_configs'][index_name]['config']}, "
            f"{EVAL_MEASURE}={cache['best_configs'][index_name]['score']:.4f}). "
            f"Skipping. Pass force=True to retune."
        )
        sync_lm_state_from_cache()
        return

    # Drop any prior rows for this index (re-tuning with force=True).
    cache["tuning_results"] = [
        r for r in cache["tuning_results"] if r.get("index") != index_name
    ]

    best_score  = -1
    best_config = None

    for step, (c) in enumerate(configs, start=1):
        print(
            f"[{step:>2}/{total}] index={index_name} c={c}",
            flush=True
        )

        model = pt.terrier.Retriever(
            index,
            wmodel="Hiemstra_LM",
            controls={
                "c": c
            }
        )
        score = evaluate_model_for_tuning(
            model, train_queries, train_qrels, measure=EVAL_MEASURE
        )

        cache["tuning_results"].append({
            "index": index_name,
            "model": "Hiemstra_LM",
            "k1":    None,
            "b":     None,
            "c":     c,
            EVAL_MEASURE: score
        })

        if score > best_score:
            best_score  = score
            best_config = {"c": c}

    cache["best_configs"][index_name] = {
        "config": best_config,
        "score":  best_score
    }

    with open(lm_cache_path, "w") as f:
        json.dump(cache, f, indent=4)

    print(f"Saved Hiemstra_LM tuning for {index_name} to {lm_cache_path}")
    sync_lm_state_from_cache()


if lm_cache_path.exists():
    sync_lm_state_from_cache()
    print(f"Loaded existing LM cache from {lm_cache_path}")
else:
    print(
        f"No LM cache yet at {lm_cache_path}.\n"
        f"Available indexes to tune: {list(indices.keys())}\n"
        f"Example:  tune_lm_for_index('stopwords')"
    )

No LM cache yet at results/lm_tuning_results.json.
Available indexes to tune: ['stopwords', 'stop_stem', 'none', 'stem']
Example:  tune_lm_for_index('stopwords')


In [60]:
tune_lm_for_index("stopwords")

[ 1/6] index=stopwords c=0.05
[ 2/6] index=stopwords c=0.1
[ 3/6] index=stopwords c=0.15
[ 4/6] index=stopwords c=0.2
[ 5/6] index=stopwords c=0.3
[ 6/6] index=stopwords c=0.5
Saved Hiemstra_LM tuning for stopwords to results/lm_tuning_results.json


In [62]:
tune_lm_for_index("stop_stem")

[ 1/6] index=stop_stem c=0.05
[ 2/6] index=stop_stem c=0.1
[ 3/6] index=stop_stem c=0.15
[ 4/6] index=stop_stem c=0.2
[ 5/6] index=stop_stem c=0.3
[ 6/6] index=stop_stem c=0.5
Saved Hiemstra_LM tuning for stop_stem to results/lm_tuning_results.json


In [64]:
tuning_results_df = pd.DataFrame(tuning_results)
tuning_results_df = tuning_results_df.sort_values(
    by=["index", "model", EVAL_MEASURE],
    ascending=[True, True, False]
)
tuning_results_df

,index,model,k1,b,c,ndcg_cut_10
32,stop_stem,BM25,0.9,0.60,NaN,0.451592
33,stop_stem,BM25,0.9,0.75,NaN,0.450662
37,stop_stem,BM25,1.2,0.60,NaN,0.450233
28,stop_stem,BM25,0.6,0.75,NaN,0.449439
38,stop_stem,BM25,1.2,0.75,NaN,0.449176
...,...,...,...,...,...,...
51,stopwords,Hiemstra_LM,NaN,NaN,0.10,0.434171
52,stopwords,Hiemstra_LM,NaN,NaN,0.15,0.434171
53,stopwords,Hiemstra_LM,NaN,NaN,0.20,0.434171
54,stopwords,Hiemstra_LM,NaN,NaN,0.30,0.434171


In [65]:
best_summary_rows = []

for index_name, model_dict in best_models.items():
    for model_name, info in model_dict.items():
        best_summary_rows.append({
            "index": index_name,
            "model": model_name,
            "best_config": info["config"],
            f"best_{EVAL_MEASURE}": round(info["score"], 4)
        })

best_summary_df = pd.DataFrame(best_summary_rows)
best_summary_df = best_summary_df.sort_values(
    by=f"best_{EVAL_MEASURE}",
    ascending=False
)

best_summary_df

,index,model,best_config,best_ndcg_cut_10
2,stop_stem,BM25,"{'k1': 0.9, 'b': 0.6}",0.4516
0,stopwords,BM25,"{'k1': 0.9, 'b': 0.6}",0.4434
3,stop_stem,Hiemstra_LM,{'c': 0.05},0.4411
1,stopwords,Hiemstra_LM,{'c': 0.05},0.4342


In [ ]:
REPORT_METRICS = [
    "ndcg_cut_5",  # ndcg_5
    "ndcg_cut_10", # ndcg_10
    "ndcg_cut_20", # ndcg_20
    "recip_rank",  # mrr (mean recipropal rank)
    "P_5",         # precision_5
    "P_10",        # precision_10
    "P_20",        # precision_20
    "recall_5",    # recall_5
    "recall_10",   # recall_10
    "recall_20",   # recall_20
    "mrt"          # mean response time
]

In [67]:
report_models = []
report_model_names = []
report_model_configs = {}

for index_name, model_dict in best_models.items():
    for model_name, info in model_dict.items():
        safe_model_name = f"{index_name}_{model_name}"

        report_models.append(info["model"])
        report_model_names.append(safe_model_name)

        report_model_configs[safe_model_name] = {
            "index": index_name,
            "ranking_model": model_name,
            "tuned_parameters": info["config"]
        }

report_model_names

['stopwords_BM25',
 'stopwords_Hiemstra_LM',
 'stop_stem_BM25',
 'stop_stem_Hiemstra_LM']

In [ ]:
trec_run_dir = Path("./results/trec_runs")
trec_run_dir.mkdir(parents=True, exist_ok=True)

final_eval_cache_path = Path("./results/final_eval_df.csv")

if final_eval_cache_path.exists():
    print(f"Loading cached final evaluation results from {final_eval_cache_path}")

    final_eval_df = pd.read_csv(final_eval_cache_path)

else:
    print("No cached final evaluation results found. Running trec-eval evaluation...")

    final_eval_df = pt.Experiment(
        report_models,
        train_queries,
        train_qrels,
        eval_metrics=REPORT_METRICS,
        names=report_model_names,
        save_dir=str(trec_run_dir),
        save_mode="overwrite",
        save_format="trec",
        filter_by_qrels=True,
        round=4
    )

    final_eval_df.to_csv(final_eval_cache_path, index=False)

    print(f"Saved final evaluation results to {final_eval_cache_path}")

final_eval_df

No cached final evaluation results found. Running trec-eval evaluation...
Saved final evaluation results to results/final_eval_df.csv


,name,recip_rank,P_5,P_10,P_20,recall_5,recall_10,recall_20,ndcg_cut_5,ndcg_cut_10,ndcg_cut_20,mrt
0,stopwords_BM25,0.4700,0.1700,0.1040,0.0602,0.4738,0.5841,0.6801,0.4036,0.4434,0.4699,287717.6139
1,stopwords_Hiemstra_LM,0.4637,0.1670,0.1031,0.0597,0.4632,0.5759,0.6727,0.3933,0.4342,0.4608,278394.9708
2,stop_stem_BM25,0.4792,0.1727,0.1050,0.0607,0.4816,0.5894,0.6849,0.4125,0.4516,0.4778,281575.4385
3,stop_stem_Hiemstra_LM,0.4708,0.1695,0.1048,0.0606,0.4701,0.5851,0.6815,0.3997,0.4411,0.4676,284441.9503
